In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, default_convert
from torch.nn.utils.rnn import pad_sequence
from typing import List
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import pandas as pd
import os
from sklearn.metrics import confusion_matrix
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR


In [2]:
from models.segment_break_detection.utils.dataset import BreakDataset
from models.segment_break_detection.utils.cnn_rnn import CNNRNN

# AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
# LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
shared_dir = "../../data/clean"
AUDIO_DIR = f"{shared_dir}/audio/vocals"
LABEL_DIR = f"{shared_dir}/segment_breaks"

interval_width = 20  # ms

MODEL_PATH = 'best_model.pth'

In [3]:
class TemporalCNN(nn.Module):
    def __init__(self, input_channels=32):
        super(TemporalCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64),
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x, mask=None):
        # x shape: (batch_size, time_steps, features)
        x = x.transpose(1, 2)  # Convert to (batch_size, features, time_steps)
        x = self.conv_layers(x)
        x = x.transpose(1, 2)  # Convert back to (batch_size, time_steps, features)
        x = self.fc(x)
        x = x.squeeze(-1)
        return x

def custom_collate(batch):
    # Separate spectrograms and labels
    spectrograms, labels = zip(*batch)

    # Pad spectrograms and labels
    padded_specs = pad_sequence(spectrograms, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=0)

    # Create mask for padded sequences
    mask = (padded_specs != 0).any(dim=-1).float()

    return padded_specs, padded_labels, mask

In [4]:
dataset = BreakDataset(AUDIO_DIR, LABEL_DIR, interval_width=interval_width, include_index=False)
indices = list(range(len(dataset)))
train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42)

# Create data loaders
train_loader = DataLoader(
    [dataset[i] for i in train_indices],
    batch_size=8,
    shuffle=True,
    collate_fn=custom_collate
)
test_loader = DataLoader(
    [dataset[i] for i in test_indices],
    batch_size=8,
    shuffle=False,
    collate_fn=custom_collate
)

Loading dataset with include_index=False...


In [24]:
import copy

# Training setup
def train_model(train_loader=train_loader, test_loader=test_loader, num_epochs=30):
    # Initialize model, loss function, and optimizer
    device = torch.device('cuda')
    model = TemporalCNN().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

    best_val_accuracy = 0.0
    best_model_state = copy.deepcopy(model.state_dict())
    patience = 3
    trigger_times = 0

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for batch_idx, (specs, labels, mask) in enumerate(train_loader):
            specs = specs.to(device).float()
            labels = labels.to(device).float()
            mask = mask.to(device)

            optimizer.zero_grad()
            outputs = model(specs)

            # Compute loss with masking
            loss = criterion(outputs, labels)
            loss = (loss * mask).mean()

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f'Epoch: {epoch+1}/{num_epochs}, Batch: {batch_idx}, Loss: {loss.item():.4f}')

        # Validation phase
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for specs, labels, mask in test_loader:
                specs = specs.to(device).float()
                labels = labels.to(device).float()
                mask = mask.to(device)

                outputs = model(specs)

                # Compute loss with masking
                loss = criterion(outputs, labels)
                loss = (loss * mask).mean()
                val_loss += loss.item()

                # Compute accuracy with masking
                predictions = (torch.sigmoid(outputs) > 0.5).float()
                correct += ((predictions == labels) * mask).sum().item()
                total += mask.sum().item()

        epoch_val_accuracy = 100 * correct / total
        print(f'Epoch: {epoch+1}/{num_epochs}')
        print(f'Training Loss: {train_loss/len(train_loader):.4f}')
        print(f'Validation Loss: {val_loss/len(test_loader):.4f}')
        print(f'Validation Accuracy: {epoch_val_accuracy:.2f}%')
        print('-' * 60)

        if epoch_val_accuracy > best_val_accuracy:
            best_val_accuracy = epoch_val_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("Early stopping!")
                model.load_state_dict(best_model_state)
                return

if __name__ == "__main__":
    train_model()


Epoch: 1/30, Batch: 0, Loss: 0.5628
Epoch: 1/30
Training Loss: 0.5628
Validation Loss: 0.4972
Validation Accuracy: 89.95%
------------------------------------------------------------
Epoch: 2/30, Batch: 0, Loss: 0.6996
Epoch: 2/30
Training Loss: 0.6996
Validation Loss: 0.5985
Validation Accuracy: 89.07%
------------------------------------------------------------
Epoch: 3/30, Batch: 0, Loss: 0.5756
Epoch: 3/30
Training Loss: 0.5756
Validation Loss: 0.4389
Validation Accuracy: 89.87%
------------------------------------------------------------
Epoch: 4/30, Batch: 0, Loss: 0.4917
Epoch: 4/30
Training Loss: 0.4917
Validation Loss: 0.3799
Validation Accuracy: 89.95%
------------------------------------------------------------
Early stopping!


In [6]:
# load saved model and make confusion matrix (since labels are boolean)
device = torch.device('cuda')
model = TemporalCNN().to(device)
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for specs, labels, mask in test_loader:
        specs = specs.to(device).float()
        labels = labels.to(device).float()
        mask = mask.to(device)

        outputs = model(specs)

        # Compute accuracy with masking
        predictions = (torch.sigmoid(outputs) > 0.5).float()
        y_true.extend(labels[mask.bool()].cpu().numpy())
        y_pred.extend(predictions[mask.bool()].cpu().numpy())
        nn.Linear(64, 1)
y_true = np.array(y_true)
y_pred = np.array(y_pred)

conf_matrix = confusion_matrix(y_true, y_pred)
print(conf_matrix)

/tmp/ipykernel_98415/3207522955.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH))


RuntimeError: Error(s) in loading state_dict for TemporalCNN:
	Missing key(s) in state_dict: "conv_layers.0.weight", "conv_layers.0.bias", "conv_layers.2.weight", "conv_layers.2.bias", "conv_layers.2.running_mean", "conv_layers.2.running_var", "conv_layers.3.weight", "conv_layers.3.bias", "conv_layers.5.weight", "conv_layers.5.bias", "conv_layers.5.running_mean", "conv_layers.5.running_var", "conv_layers.6.weight", "conv_layers.6.bias", "conv_layers.8.weight", "conv_layers.8.bias", "conv_layers.8.running_mean", "conv_layers.8.running_var", "fc.weight", "fc.bias". 
	Unexpected key(s) in state_dict: "conv.0.weight", "conv.0.bias", "conv.1.weight", "conv.1.bias", "conv.1.running_mean", "conv.1.running_var", "conv.1.num_batches_tracked", "conv.3.weight", "conv.3.bias", "conv.4.weight", "conv.4.bias", "conv.4.running_mean", "conv.4.running_var", "conv.4.num_batches_tracked", "rnn.weight_ih_l0", "rnn.weight_hh_l0", "rnn.bias_ih_l0", "rnn.bias_hh_l0", "rnn.weight_ih_l0_reverse", "rnn.weight_h